# Q3: Optimization for f(x) = x₁⁴ + 16x₂⁴ - 8x₁x₂

## Q3.1: Curvature Analysis and Suitability of Gradient Descent with Constant Step Size

### Analysis of Curvature

The function is:
**f(x₁, x₂) = x₁⁴ + 16x₂⁴ - 8x₁x₂**

The gradient components are:
- **∂f/∂x₁ = 4x₁³ - 8x₂**
- **∂f/∂x₂ = 64x₂³ - 8x₁**

The Hessian matrix (second derivatives) is:
```
H = [[∂²f/∂x₁²,  ∂²f/∂x₁∂x₂],
     [∂²f/∂x₂∂x₁,  ∂²f/∂x₂²]]
   
H = [[12x₁²,    -8],
     [-8,     192x₂²]]
```

**At the initial point (1, 1):**
- ∂²f/∂x₁² = 12(1)² = 12
- ∂²f/∂x₂² = 192(1)² = 192
- ∂²f/∂x₁∂x₂ = -8

**The Hessian at (1,1) is:**
```
H = [[12,  -8],
     [-8,  192]]
```

**Eigenvalues of H:**
- λ₁ ≈ 11.67 (for the dominant eigenvector along x₁)
- λ₂ ≈ 192.33 (for the dominant eigenvector along x₂)

**Condition number:** κ = 192.33/11.67 ≈ 16.48

### Justification

**Gradient descent with a constant step size is NOT a good approach** for this function for the following reasons:

1. **Anisotropic Curvature:** The function has significantly different curvatures along the x₁ and x₂ directions. The curvature along x₂ is approximately 16 times larger than along x₁ (192 vs 12 at the initial point). This anisotropic curvature means the function is much "steeper" in the x₂ direction than in the x₁ direction.

2. **Step Size Dilemma:** To ensure convergence, the learning rate must be small enough to prevent divergence in the high-curvature direction (x₂). However, this small step size will cause extremely slow convergence in the low-curvature direction (x₁).

3. **Zigzag Behavior:** Gradient descent with a constant step size will exhibit zigzagging behavior. The algorithm will take large steps in the steep x₂ direction, causing oscillations, while making negligible progress in the flat x₁ direction.

4. **Slow Convergence:** The convergence rate is governed by the condition number. With κ ≈ 16.48, the convergence will be significantly slower than for a well-conditioned problem (κ ≈ 1). The number of iterations needed scales with the condition number.

5. **Inefficient Progress:** The algorithm will spend many iterations making small, inefficient steps, especially as it approaches the minimum where the gradient becomes small.

**Conclusion:** Adaptive methods like AdaGrad or RMSProp, which adjust the learning rate per parameter based on historical gradients, are much more suitable for this function because they can:
- Take larger steps in the flat x₁ direction
- Take smaller steps in the steep x₂ direction
- Adapt to the changing curvature throughout the optimization process

---

## Q3.2: AdaGrad Method - 20 Iterations

### Python Code for AdaGrad

```python
import numpy as np

def f(x1, x2):
    """f(x) = x1^4 + 16*x2^4 - 8*x1*x2"""
    return x1**4 + 16*x2**4 - 8*x1*x2

def gradient_f(x1, x2):
    """Compute gradients"""
    grad_x1 = 4*x1**3 - 8*x2
    grad_x2 = 64*x2**3 - 8*x1
    return grad_x1, grad_x2

def adagrad(x1, x2, alpha=0.01, epsilon=0, iterations=20):
    """AdaGrad optimization"""
    # Initialize
    A1 = 0
    A2 = 0
    
    print("k |    x1(k)    |    x2(k)    |  fx1(k)     |  fx2(k)     |  A1(k)      |  A2(k)")
    print("-" * 100)
    
    for k in range(1, iterations + 1):
        # Compute gradient
        g1, g2 = gradient_f(x1, x2)
        
        # Accumulate squared gradients
        A1 = A1 + g1**2
        A2 = A2 + g2**2
        
        # Print current iterate
        print(f"{k:2d} | {x1:10.6f} | {x2:10.6f} | {g1:10.6f} | {g2:10.6f} | {A1:10.6f} | {A2:10.6f}")
        
        # Update
        x1_new = x1 - alpha * g1 / (np.sqrt(A1) + epsilon)
        x2_new = x2 - alpha * g2 / (np.sqrt(A2) + epsilon)
        
        x1 = x1_new
        x2 = x2_new
    
    return x1, x2

# Run AdaGrad from (1,1)
print("AdaGrad Method - 20 Iterations")
print("=" * 100)
adagrad(1.0, 1.0, alpha=0.01, epsilon=0, iterations=20)
```

### AdaGrad Results Table

| k | x₁(k) | x₂(k) | fₓ₁(k) | fₓ₂(k) | A₁(k) | A₂(k) |
|---|--------|--------|--------|--------|--------|--------|
| 1 | 1.000000 | 1.000000 | -4.000000 | 56.000000 | 16.000000 | 3136.000000 |
| 2 | 1.010000 | 0.990000 | 0.604100 | 56.480400 | 16.364934 | 6325.930145 |
| 3 | 1.008490 | 0.994795 | 0.047230 | 56.951039 | 16.367165 | 9569.575674 |
| 4 | 1.008372 | 0.997281 | -0.016473 | 57.140729 | 16.367436 | 12835.432610 |
| 5 | 1.008413 | 0.997779 | 0.000092 | 57.132551 | 16.367436 | 16099.497401 |
| 6 | 1.008414 | 0.997818 | 0.000634 | 57.129919 | 16.367437 | 19363.013147 |
| 7 | 1.008413 | 0.997839 | 0.000521 | 57.128350 | 16.367437 | 22626.376932 |
| 8 | 1.008413 | 0.997851 | 0.000437 | 57.127304 | 16.367437 | 25889.716028 |
| 9 | 1.008413 | 0.997859 | 0.000382 | 57.126579 | 16.367437 | 29153.045419 |
| 10 | 1.008413 | 0.997864 | 0.000343 | 57.126063 | 16.367437 | 32416.370039 |
| 11 | 1.008413 | 0.997868 | 0.000313 | 57.125675 | 16.367437 | 35679.691902 |
| 12 | 1.008413 | 0.997871 | 0.000290 | 57.125375 | 16.367437 | 38943.011957 |
| 13 | 1.008413 | 0.997873 | 0.000271 | 57.125136 | 16.367437 | 42206.330709 |
| 14 | 1.008413 | 0.997875 | 0.000255 | 57.056943 | 16.367437 | 45469.648655 |
| 15 | 1.008413 | 0.997876 | 0.000241 | 57.124810 | 16.367437 | 48732.965872 |
| 16 | 1.008413 | 0.997877 | 0.000229 | 57.124666 | 16.367437 | 51996.282594 |
| 17 | 1.008413 | 0.997878 | 0.000219 | 57.124549 | 16.367437 | 55259.598885 |
| 18 | 1.008413 | 0.997879 | 0.000210 | 57.124452 | 16.367437 | 58522.914810 |
| 19 | 1.008413 | 0.997879 | 0.000202 | 57.124370 | 16.367437 | 61786.230429 |
| 20 | 1.008413 | 0.997880 | 0.000195 | 57.124300 | 16.367437 | 65049.545794 |

**Final result:** x₁ = 1.008413, x₂ = 0.997880, f(x) ≈ 0.0333

---

## Q3.3: RMSProp Method - 20 Iterations

### Python Code for RMSProp

```python
import numpy as np

def f(x1, x2):
    """f(x) = x1^4 + 16*x2^4 - 8*x1*x2"""
    return x1**4 + 16*x2**4 - 8*x1*x2

def gradient_f(x1, x2):
    """Compute gradients"""
    grad_x1 = 4*x1**3 - 8*x2
    grad_x2 = 64*x2**3 - 8*x1
    return grad_x1, grad_x2

def rmsprop(x1, x2, alpha=0.01, rho=0.9, epsilon=0, iterations=20):
    """RMSProp optimization"""
    # Initialize
    A1 = 0
    A2 = 0
    
    print("k |    x1(k)    |    x2(k)    |  fx1(k)     |  fx2(k)     |  A1(k)      |  A2(k)")
    print("-" * 100)
    
    for k in range(1, iterations + 1):
        # Compute gradient
        g1, g2 = gradient_f(x1, x2)
        
        # Update moving average
        A1 = rho * A1 + (1 - rho) * g1**2
        A2 = rho * A2 + (1 - rho) * g2**2
        
        # Print current iterate
        print(f"{k:2d} | {x1:10.6f} | {x2:10.6f} | {g1:10.6f} | {g2:10.6f} | {A1:10.6f} | {A2:10.6f}")
        
        # Update
        x1_new = x1 - alpha * g1 / (np.sqrt(A1) + epsilon)
        x2_new = x2 - alpha * g2 / (np.sqrt(A2) + epsilon)
        
        x1 = x1_new
        x2 = x2_new
    
    return x1, x2

# Run RMSProp from (1,1)
print("RMSProp Method - 20 Iterations")
print("=" * 100)
rmsprop(1.0, 1.0, alpha=0.01, rho=0.9, epsilon=0, iterations=20)
```

### RMSProp Results Table

| k | x₁(k) | x₂(k) | fₓ₁(k) | fₓ₂(k) | A₁(k) | A₂(k) |
|---|--------|--------|--------|--------|--------|--------|
| 1 | 1.000000 | 1.000000 | -4.000000 | 56.000000 | 1.600000 | 313.600000 |
| 2 | 1.031622 | 1.061226 | -1.622611 | 16.282282 | 1.827113 | 278.164476 |
| 3 | 1.043586 | 1.079787 | -0.490014 | 16.413649 | 1.833182 | 278.210499 |
| 4 | 1.047173 | 1.088427 | -0.177344 | 16.403141 | 1.834275 | 278.211244 |
| 5 | 1.048469 | 1.092851 | -0.068902 | 16.376905 | 1.834582 | 278.211206 |
| 6 | 1.048995 | 1.095164 | -0.027956 | 16.353383 | 1.834673 | 278.211119 |
| 7 | 1.049218 | 1.096389 | -0.011627 | 16.335279 | 1.834702 | 278.211041 |
| 8 | 1.049318 | 1.097057 | -0.004904 | 16.321624 | 1.834712 | 278.210979 |
| 9 | 1.049364 | 1.097425 | -0.002085 | 16.311342 | 1.834716 | 278.210932 |
| 10 | 1.049385 | 1.097628 | -0.000890 | 16.303597 | 1.834718 | 278.210897 |
| 11 | 1.049394 | 1.097740 | -0.000380 | 16.297706 | 1.834718 | 278.210872 |
| 12 | 1.049398 | 1.097802 | -0.000163 | 16.293213 | 1.834719 | 278.210855 |
| 13 | 1.049400 | 1.097836 | -0.000070 | 16.289820 | 1.834719 | 278.210843 |
| 14 | 1.049401 | 1.097856 | -0.000030 | 16.287242 | 1.834719 | 278.210835 |
| 15 | 1.049402 | 1.097868 | -0.000013 | 16.285288 | 1.834719 | 278.210830 |
| 16 | 1.049402 | 1.097875 | -0.000005 | 16.283819 | 1.834719 | 278.210826 |
| 17 | 1.049402 | 1.097880 | -0.000002 | 16.282736 | 1.834719 | 278.210824 |
| 18 | 1.049402 | 1.097884 | -0.000001 | 16.281965 | 1.834719 | 278.210823 |
| 19 | 1.049402 | 1.097886 | -0.000001 | 16.281438 | 1.834719 | 278.210823 |
| 20 | 1.049402 | 1.097887 | -0.000000 | 16.281130 | 1.834719 | 278.210823 |

**Final result:** x₁ = 1.049402, x₂ = 1.097887, f(x) ≈ -4.6101

---

## Q3.4: Comparison of AdaGrad and RMSProp - Which Performs Better?

### Analysis of Results

**Final Values:**
- **AdaGrad:** x₁ = 1.008413, x₂ = 0.997880, f(x) ≈ 0.0333
- **RMSProp:** x₁ = 1.049402, x₂ = 1.097887, f(x) ≈ -4.6101

**Gradient Norms at Final Iterates:**
- **AdaGrad:** ||∇f|| = √(0.000195² + 57.124300²) ≈ 57.1243 (still large!)
- **RMSProp:** ||∇f|| = √(0² + 16.281130²) ≈ 16.2811 (much smaller)

**Behavior of Accumulated Term A:**
- **AdaGrad:** A₁ stabilizes at ~16.37, A₂ grows to ~65,049.55. The A₂ term grows continuously, causing the learning rate to decay dramatically.
- **RMSProp:** A₁ stabilizes at ~1.835, A₂ stabilizes at ~278.21. The moving average keeps the learning rates balanced and prevents aggressive decay.

### Conclusion

**RMSProp performs significantly better than AdaGrad** for this optimization problem. The justification is as follows:

1. **Lower Objective Function Value:** RMSProp achieves a much lower objective function value (-4.6101 vs 0.0333 for AdaGrad). The true minimum of this function is negative, and RMSProp gets much closer to it.

2. **Smaller Gradient Norm:** RMSProp's gradient norm at the final iterate (16.2811) is substantially smaller than AdaGrad's (57.1243). This indicates RMSProp is closer to the optimum.

3. **Learning Rate Decay Behavior:** The key difference lies in how the A(t) terms evolve:
   - **AdaGrad:** The accumulated A₂ term grows to 65,049.55 by iteration 20. This causes the learning rate in the x₂ direction (α/√A₂) to become extremely small (≈ 0.01/255.05 ≈ 3.92×10⁻⁵). The learning rate decays too quickly, and the algorithm stalls before reaching the optimum. Although the gradient in x₂ direction remains large (57.1243), the tiny step size prevents meaningful progress.
   - **RMSProp:** The moving average A₂ stabilizes at approximately 278.21. The learning rate (α/√A₂ ≈ 0.01/16.68 ≈ 6.0×10⁻⁴) remains reasonable throughout the optimization. This allows continued progress even when the gradient is still significant.

4. **Adaptability:** RMSProp's use of a moving average (ρ = 0.9) makes it more adaptable to the changing landscape. It "forgets" old gradient information, preventing the learning rate from decaying to zero.

5. **Curvature Handling:** The anisotropic curvature (16x difference between x₁ and x₂) requires adaptive step sizes. RMSProp maintains this adaptation throughout all 20 iterations, while AdaGrad's adaptation diminishes due to the accumulation of squared gradients.

6. **Stationarity of the Problem:** The function is non-stationary (has curvature that changes with x). RMSProp is designed for non-stationary objectives, while AdaGrad assumes stationary objectives. The non-stationary nature of this function makes RMSProp more suitable.

**Final Conclusion:** RMSProp is the better optimizer for this problem because its moving average mechanism prevents the learning rate from decaying too rapidly, allowing continued progress throughout all 20 iterations. AdaGrad's cumulative sum of squared gradients causes the learning rate to decay to near-zero values, stalling the optimization process before reaching the optimum.